# Overview

The purpose of this notebook is to understand the logic of `fetch_dsd_schema` developed by Tony, then to improve it and make it more robust and finally to extend it to dataflows.

**Note: This notebook is for development purposes and it is not intended to be used in production nor to be executed sequentially.**

# Setup

In [ ]:
# import pysdmx as px

In [ ]:
from pysdmx.io.format import StructureFormat # To extract json format
from pysdmx.api import fmr # CLient to connect to FMR
from urllib.parse import urljoin

# parse_artefact_id

## line by line

In [ ]:
artefact_id = "WB.DATA360:DS_DATA360(1.3)"

In [ ]:
agency, rest = artefact_id.split(":")
print(agency)
print(rest)

In [ ]:
id, version = rest.split("(")
version = version.rstrip(")")
print(id)
print(version)

## function

In [ ]:
def parse_artefact_id(artefact_id):
    """Parses artefact identifier (DSD, Dataflow, Codelist, etc) into its components: agency, id and version.

    Args:
        artefact_id (str): The identifier of the artefact, typically in the format "agency:id(version)".

    Returns:
        tuple: A tuple containing the agency, id, and version.

    Raises:
        ValueError: If the artefact_id is not in the expected format.
    """

    try:
        agency, rest = artefact_id.split(":", 1)
        id_part, version_part = rest.split("(", 1)
        version = version_part.rstrip(")")
        return agency, id_part, version
    except Exception:
        raise ValueError("Invalid artefact_id format. Expected format: 'agency:id(version)'")

# fetch_schema

## Line by line

In [ ]:
dsd_id = "WB.DATA360:DS_DATA360(1.3)"

In [ ]:
# # If I import only the package, I cannot access this object. 
# # I need to access the class directly.
# px.io.format.StructureFormat.FUSION_JSON

In [ ]:
format = StructureFormat.FUSION_JSON
format

In [ ]:
# fmr_url = fmr_params[env]["url"]
fmr_url = 'https://fmrqa.worldbank.org/'
fmr_url

In [ ]:
# Ensure the URL is syntactically valid
base_url = urljoin(fmr_url, "/FMR/sdmx/v2/")
base_url

In [ ]:
client = fmr.RegistryClient(
        base_url,
        format=format,
    )
client

In [ ]:
agency, id, version = parse_artefact_id(dsd_id)
print(agency)
print(id)
print(version)

### DSD

In [ ]:
# ## Checking URLS for DSDs and Dataflows
# # DSDs
# https://fmrqa.worldbank.org/FMR/sdmx/v2/structure/datastructure/WB/IFPRI_ASTI/1.0
# # Dataflows
# https://fmrqa.worldbank.org/FMR/sdmx/v2/structure/dataflow/WB/DF_IFPRI_ASTI/1.0

In [ ]:
schema = client.get_schema("datastructure", agency, id, version)
print(schema)

In [ ]:
dir(schema)

### Dataflow

In [ ]:
dataflow_id = "WB.DATA360:DF_D360_WB_WDI(1.0)"

In [ ]:
# ## Checking URLS for DSDs and Dataflows
# # DSDs
# https://fmrqa.worldbank.org/FMR/sdmx/v2/structure/datastructure/WB/IFPRI_ASTI/1.0
# # Dataflows
# https://fmrqa.worldbank.org/FMR/sdmx/v2/structure/dataflow/WB/DF_IFPRI_ASTI/1.0

In [ ]:
agency, id, version = parse_artefact_id(dataflow_id)

In [ ]:
schema = client.get_schema("dataflow", agency, id, version)
print(schema)

In [ ]:
dir(schema)

### provisionagreement

In [ ]:
provision_id = "WB.TEST:DF_CSC_EN_FSH_SUST_ZS_WB_TEST_DP_ENV(1.0)"

In [ ]:
# ## Checking URLS for DSDs and Dataflows
# # DSDs
# https://fmrqa.worldbank.org/FMR/sdmx/v2/structure/datastructure/WB/IFPRI_ASTI/1.0
# # Dataflows
# https://fmrqa.worldbank.org/FMR/sdmx/v2/structure/dataflow/WB/DF_IFPRI_ASTI/1.0

In [ ]:
agency, id, version = parse_artefact_id(provision_id)

In [ ]:
schema = client.get_schema("provisionagreement", agency, id, version)
print(schema)

## Function

In [ ]:
def fetch_schema(
		base_url:str,
		artefact_id: str,
		context: str = "datastructure"):
	"""Fetches the schema of a specified artefact from an SDMX registry.
	
	Args:
		base_url (str): The base URL of the FMR.
		artefact_id (str): The identifier of the artefact, typically in the format "agency:id(version)".
		context (str, optional): The type of artefact to fetch. Defaults to "datastructure". It can also be "dataflow" and "provisionagreement".
	Returns:
		schema: The fetched schema object.
	"""
	format = StructureFormat.FUSION_JSON

	# Ensure the URL is syntactically valid
	base_url = urljoin(base_url, "/FMR/sdmx/v2/")

	# Initialize the client
	client = fmr.RegistryClient(
        base_url,
        format=format,
    )

	# Parse the artefact ID
	agency, id, version = parse_artefact_id(artefact_id)

	# Fetch the schema
	schema = client.get_schema(context, agency, id, version)
	
	return schema